In [1]:
import duckdb

In [2]:
conn = duckdb.connect()

In [3]:
conn.execute('/*FORCE*/ INSTALL webbed FROM community;')
conn.execute('LOAD webbed;')

## WEBSCRAPING EXTENTIONS

In [4]:
sql = """COPY(
    SELECT html::VARCHAR 
    FROM READ_HTML_OBJECTS('https://duckdb.org/community_extensions/list_of_extensions')
) TO 'extensions.html' (HEADER FALSE, DELIM '', ESCAPE '', QUOTE '');"""
conn.execute(sql)

In [5]:
sql = """SELECT content/*, filename, size, last_modified*/
FROM READ_TEXT('extensions.html') as t;"""
conn.sql(sql).show()


┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [7]:
sql = """SET VARIABLE html_content = (
    SELECT content
    FROM READ_TEXT('extensions.html')
);"""
conn.execute(sql)

In [ ]:
sql = """SELECT GETVARIABLE('html_content');"""
#conn.sql(sql).show()

In [8]:
sql = """SELECT REGEXP_REPLACE(REGEXP_REPLACE(columns[1], '^\\s+|\\s+$|', ''), '^\\s+|\\s+$', '') AS Name
    , REGEXP_REPLACE(REGEXP_REPLACE(columns[2], '^\\s+|\\s+$', ''), '^\\s+|\\s+$', '') AS GitHub
    , REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(columns[3], '^\\s+|\\s+$', ''), '^\\s+|\\s+$', ''), '^\\s+|\\s+$', '') AS Description
    , CURRENT_DATE AS ExtractDate
FROM HTML_EXTRACT_TABLES(GETVARIABLE('html_content'))
OFFSET 1 /*STARTS FOR LINE 2*/;"""
conn.sql(sql).show()

┌─────────────────┬─────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┐
│      Name       │ GitHub  │                                                                                                   Description                                                                                                   │ ExtractDate │
│     varchar     │ varchar │                                                                                                     varchar                                                                                                     │    date     │
├─────────────────┼─────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [9]:
sql2 = f"""CREATE TABLE IF NOT EXISTS Extensions AS {sql}"""
conn.execute(sql2)

In [10]:
sql2 = """MERGE INTO Extensions AS TARGET
USING (
    SELECT REGEXP_REPLACE(REGEXP_REPLACE(columns[1], '^\\s+|\\s+$|', ''), '^\\s+|\\s+$', '') AS Name
        , REGEXP_REPLACE(REGEXP_REPLACE(columns[2], '^\\s+|\\s+$', ''), '^\\s+|\\s+$', '') AS GitHub
        , REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(columns[3], '^\\s+|\\s+$', ''), '^\\s+|\\s+$', ''), '^\\s+|\\s+$', '') AS Description
        , CURRENT_DATE AS ExtractDate
    FROM HTML_EXTRACT_TABLES(GETVARIABLE('html_content'))
    OFFSET 1 /*STARTS FOR LINE 2*/
) AS SOURCE
ON TARGET.Name = SOURCE.Name
WHEN MATCHED THEN
    UPDATE SET
        GitHub = SOURCE.GitHub,
        Description = SOURCE.Description,
        ExtractDate = TARGET.ExtractDate
WHEN NOT MATCHED THEN
    INSERT (Name, GitHub, Description, ExtractDate)
    VALUES (SOURCE.Name, SOURCE.GitHub, SOURCE.Description, SOURCE.ExtractDate)"""
conn.execute(sql2)

### CHECK NEW EXT

In [11]:
sql = """SELECT ExtractDate, count(*) as count FROM Extensions GROUP BY ExtractDate ORDER BY ExtractDate DESC;"""
conn.sql(sql).show()

┌─────────────┬───────┐
│ ExtractDate │ count │
│    date     │ int64 │
├─────────────┼───────┤
│ 2026-06-11  │   199 │
└─────────────┴───────┘



In [12]:
sql = """SELECT * FROM Extensions WHERE ExtractDate = '2026-06-11';"""
conn.sql(sql).show()

┌─────────────────┬─────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┐
│      Name       │ GitHub  │                                                                                                   Description                                                                                                   │ ExtractDate │
│     varchar     │ varchar │                                                                                                     varchar                                                                                                     │    date     │
├─────────────────┼─────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

### USING XML

In [13]:
sql = """COPY(
    SELECT REGEXP_REPLACE(
        content, 
        '<!doctype html>|<head[\s\S]*?</head>|<meta[^>]*>|<link[^>]*>|<img[^>]*>|<input[^>]*>|\s*defer|<svg[\s\S]*?</svg>|<script[\s\S]*?</script>|<style[\s\S]*?</style>|\&|–', 
        '', 'g'
    ) AS content
    FROM READ_TEXT('extensions.html')
) TO 'extensions.xml' (HEADER FALSE, DELIM '', ESCAPE '', QUOTE '');"""
conn.execute(sql)

In [14]:
sql = """WITH tr AS (
    SELECT UNNEST(XML_EXTRACT_ELEMENTS(xml::xml, '//html/body/main/div/div/table/tbody/tr')) AS TableContent
    FROM read_xml_objects('extensions.xml')
)
SELECT XML_EXTRACT_TEXT(TableContent, '//tr/td[1]/a')[1]    AS Name, 
    XML_EXTRACT_TEXT(TableContent, '//tr/td[1]/a/@href')[1] AS Route, 
    XML_EXTRACT_TEXT(TableContent, '//tr/td[2]/a/@href')[1] AS Repository,
    REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(XML_EXTRACT_TEXT(TableContent, '//tr/td[3]')[1], '^\s+|\s+$', ''), '^\s+|\s+$', ''), '^\s+|\s+$', '') AS Description
FROM tr;"""
conn.sql(sql).show()

┌─────────────────┬───────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│      Name       │                         Route                         │                        Repository                         │                                                                                                   Description                                                                                                   │
│     varchar     │                        varchar                        │                          varchar                          │                                                                                                     varchar                                                                 